# Análise de Preços do Ponto de Função - 2026
Inventário de Bens Intangíveis da SEPLAG/MT. Amostra: 9 preços de serviços em Java medidos em ponto de função (Compras.gov.br, 2025-2026).
Sequência igual à dos notebooks de 2025 (`Analise_Precos_PF-2025` e `Segmentacao_Precos_Atualizados_K_Means`).

In [1]:
# ==========================================
# CÉLULA 0: Import dos Pacotes e Bibliotecas
# ==========================================
import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
pd.options.display.float_format = '{:,.2f}'.format

In [2]:
# ==========================================
# CÉLULA 1: Carregamento dos Dados (mapa comparativo de preços)
# ==========================================
mapa = pd.read_csv('../4. Mapa Comparativo/Precos_Ponto_Funcao_2025-2026.csv', sep=';', encoding='utf-8-sig')
amostra = mapa[mapa['classificacao'].str.startswith('AMOSTRA A')].copy()
amostra = amostra[['id', 'uasg_nome', 'uf', 'fornecedor', 'modalidade_nome', 'data_compra', 'preco_unitario_pf',
                   'mes_base_preco', 'fator_ipca', 'preco_atualizado_pf', 'dentro_13_meses', 'disputa_de_preco']]
lista_precos = amostra['preco_unitario_pf'].tolist()              # valores nominais
lista_precos_atualizados = amostra['preco_atualizado_pf'].tolist()  # corrigidos pelo IPCA até 10/2026
print("Preços nominais:    ", lista_precos)
print("Preços atualizados: ", lista_precos_atualizados)
print(f"Total de registros: {len(amostra)}")
amostra

Preços nominais:     [555.0, 565.15, 1060.0, 735.0, 750.0, 2192.73, 1264.22, 538.0, 785.0]
Preços atualizados:  [601.41, 597.0, 1114.16, 769.02, 783.3, 2290.1, 1316.01, 558.2, 790.01]
Total de registros: 9


,id,uasg_nome,uf,fornecedor,modalidade_nome,data_compra,preco_unitario_pf,mes_base_preco,fator_ipca,preco_atualizado_pf,dentro_13_meses,disputa_de_preco
0,1,CASA DA MOEDA DO BRASIL/MF,RJ,M.I. MONTREAL INFORMATICA S.A,Pregão - Eletrônico,2025-02-17,555.00,02/2025,1.08,601.41,Não,Sim
1,2,ERN-SECRET.DA SEGURANÇA PUBL. E DEFESA SOCIAL,RN,SIGMA DATASERV INFORMATICA S A EM RECUPERACAO ...,Inexigibilidade,2025-06-09,565.15,06/2025,1.06,597.00,Não,Não
2,3,PAGADORIA DE PESSOAL DA MARINHA - COM MARINHA,RJ,INDRA BRASIL SOLUCOES E SERVICOS TECNOLOGICOS ...,Pregão - Eletrônico,2025-08-15,"1,060.00",08/2025,1.05,"1,114.16",Não,Sim
3,4,UNIVERSIDADE FEDERAL DO CARIRI,CE,SIG SOFTWARE & CONSULTORIA EM TECNOLOGIA DA IN...,Pregão - Eletrônico,2025-11-14,735.00,11/2025,1.05,769.02,Sim,Sim
4,5,MINISTÉRIO PÚBLICO DO ESTADO DO PARÁ,PA,THEMA INFORMATICA LTDA,Inexigibilidade,2025-12-10,750.00,12/2025,1.04,783.30,Sim,Não
5,6,INCRA-SEDE/DF,DF,SERVICO FEDERAL DE PROCESSAMENTO DE DADOS (SER...,Dispensa,2025-12-23,"2,192.73",12/2025,1.04,"2,290.10",Sim,Não
6,7,TRIBUNAL DE JUSTIÇA DO ESTADO DO PARÁ,PA,SIEDOS SISTEMAS E RESULTADOS LTDA,Pregão - Eletrônico,2026-01-07,"1,264.22",01/2026,1.04,"1,316.01",Sim,Sim
7,8,SUBSECRETARIA DE ASSUNTOS ADMINISTRATIVOS,DF,IBROWSE - CONSULTORIA & INFORMATICA LTDA,Pregão - Eletrônico,2026-02-26,538.00,02/2026,1.04,558.20,Sim,Sim
8,9,SECRETARIA MUNICIPAL DA EDUCAÇÃO DE SALVADOR,BA,CONSORCIO ASTEC IV,Inexigibilidade,2026-08-20,785.00,08/2026,1.01,790.01,Sim,Não


In [3]:
# ==========================================
# CÉLULA 2: Estatísticas Descritivas (valores nominais)
# ==========================================
series_precos = pd.Series(lista_precos)
media, mediana = series_precos.mean(), series_precos.median()
print(f"Média:   {media:.2f}")
print(f"Mediana: {mediana:.2f}")
print(f"Desvio padrão: {series_precos.std():.2f}")
print(f"Coeficiente de variação: {series_precos.std()/media*100:.1f}%")
print("Moda: não há valor repetido na amostra")

Média:   938.34
Mediana: 750.00
Desvio padrão: 529.31
Coeficiente de variação: 56.4%
Moda: não há valor repetido na amostra


In [4]:
# ==========================================
# CÉLULA 3: Distribuição e Boxplot iniciais (valores nominais)
# ==========================================
plt.figure(figsize=(10, 6))
sns.histplot(series_precos, kde=True, stat="density", color="skyblue", label="Dados Reais", bins=9)
xmin, xmax = plt.xlim(); x = np.linspace(xmin, xmax, 200)
plt.plot(x, stats.norm.pdf(x, media, series_precos.std()), 'k', linewidth=2, label='Curva Normal Teórica')
plt.title('Distribuição Inicial dos Preços (nominais)'); plt.xlabel('Preço do ponto de função (R$)'); plt.legend()
plt.savefig('1_distribuicao_inicial.png', dpi=150, bbox_inches='tight'); plt.close()

plt.figure(figsize=(6, 8))
sns.boxplot(y=series_precos, color="salmon")
plt.title('Boxplot Inicial (Identificação de Outliers)'); plt.ylabel('Preço do ponto de função (R$)')
plt.savefig('2_boxplot_inicial.png', dpi=150, bbox_inches='tight'); plt.close()
print("Gráficos 1 e 2 salvos.")

Gráficos 1 e 2 salvos.


In [5]:
# ==========================================
# CÉLULA 4: Quantis e Outliers (IQR) - valores nominais
# ==========================================
def iqr(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75); a = q3 - q1
    return q1, q3, a, q1 - 1.5 * a, q3 + 1.5 * a
Q1, Q3, IQR, limite_inferior, limite_superior = iqr(series_precos)
outliers = series_precos[(series_precos < limite_inferior) | (series_precos > limite_superior)]
print(f"Q1 (25%): {Q1:.2f}\nQ3 (75%): {Q3:.2f}\nIQR: {IQR:.2f}")
print(f"Faixa aceitável: {limite_inferior:.2f} até {limite_superior:.2f}")
print(f"Outliers detectados: {len(outliers)} -> {outliers.values}")
series_limpa = series_precos[(series_precos >= limite_inferior) & (series_precos <= limite_superior)]

plt.figure(figsize=(10, 6))
sns.histplot(series_limpa, kde=True, stat="density", color="green", label="Dados Limpos", bins=8)
xmin, xmax = plt.xlim(); x = np.linspace(xmin, xmax, 200)
plt.plot(x, stats.norm.pdf(x, series_limpa.mean(), series_limpa.std()), 'k', linewidth=2, label='Nova Curva Normal')
plt.title('Distribuição de Preços (Sem Outliers)'); plt.xlabel('Preço do ponto de função (R$)'); plt.legend()
plt.savefig('3_distribuicao_limpa.png', dpi=150, bbox_inches='tight'); plt.close()
plt.figure(figsize=(6, 8))
sns.boxplot(y=series_limpa, color="lightgreen"); plt.title('Boxplot (Pós-Limpeza)'); plt.ylabel('Preço do ponto de função (R$)')
plt.savefig('4_boxplot_limpo.png', dpi=150, bbox_inches='tight'); plt.close()
print(f"Dados originais: {len(series_precos)} -> Dados sem outliers: {len(series_limpa)}")
print(f"Média sem outliers: {series_limpa.mean():.2f} | Mediana sem outliers: {series_limpa.median():.2f}")

Q1 (25%): 565.15
Q3 (75%): 1060.00
IQR: 494.85
Faixa aceitável: -177.13 até 1802.28
Outliers detectados: 1 -> [2192.73]


Dados originais: 9 -> Dados sem outliers: 8
Média sem outliers: 781.55 | Mediana sem outliers: 742.50


In [6]:
# ==========================================
# CÉLULA 5: Valores atualizados pelo IPCA (data-base 10/2026)
# ==========================================
s_at = pd.Series(lista_precos_atualizados)
Q1a, Q3a, IQRa, li_a, ls_a = iqr(s_at)
out_at = s_at[(s_at < li_a) | (s_at > ls_a)]
print(f"Média: {s_at.mean():.2f} | Mediana: {s_at.median():.2f} | Desvio padrão: {s_at.std():.2f}")
print(f"Q1: {Q1a:.2f} | Q3: {Q3a:.2f} | IQR: {IQRa:.2f} | Faixa aceitável: {li_a:.2f} até {ls_a:.2f}")
print(f"Outliers detectados: {len(out_at)} -> {out_at.values}")
plt.figure(figsize=(12, 7))
sns.histplot(s_at, kde=True, color="purple", stat="density", label='Distribuição', bins=9)
plt.axvline(s_at.mean(), color='red', linestyle='--', linewidth=2, label=f'Média: {s_at.mean():.2f}')
plt.axvline(s_at.median(), color='orange', linestyle='-', linewidth=2, label=f'Mediana: {s_at.median():.2f}')
plt.title('Distribuição dos Preços Atualizados (IPCA até 10/2026) com Medidas de Posição')
plt.xlabel('Preço do ponto de função (R$)'); plt.ylabel('Densidade'); plt.legend()
plt.savefig('5_distribuicao_estatisticas.png', dpi=150, bbox_inches='tight'); plt.close()

Média: 979.91 | Mediana: 783.30 | Desvio padrão: 551.65
Q1: 601.41 | Q3: 1114.16 | IQR: 512.75 | Faixa aceitável: -167.72 até 1883.29
Outliers detectados: 1 -> [2290.1]


In [7]:
# ==========================================
# CÉLULA 6: Segmentação K-Means (3 grupos) - função reutilizável
# ==========================================
NOMES = ['Grupo 1 (Entrada)', 'Grupo 2 (Intermediário)', 'Grupo 3 (Premium)']
def segmentar(df_in, coluna, titulo, arquivo):
    df = df_in.copy()
    km = KMeans(n_clusters=3, random_state=42, n_init=10)
    df['Cluster'] = km.fit_predict(df[[coluna]])
    ordem = df.groupby('Cluster')[coluna].mean().sort_values().index
    df['Grupo'] = df['Cluster'].map({c: NOMES[i] for i, c in enumerate(ordem)})
    resumo = df.groupby('Grupo')[coluna].agg(['count', 'min', 'max', 'mean', 'median']).reindex(NOMES)
    sil = silhouette_score(df[[coluna]], df['Cluster']) if len(df) > 3 else float('nan')
    plt.figure(figsize=(12, 6))
    cores = {NOMES[0]: '#2a9d8f', NOMES[1]: '#e9c46a', NOMES[2]: '#e76f51'}
    for g in NOMES:
        d = df[df['Grupo'] == g]
        plt.scatter(d[coluna], [0] * len(d), s=260, color=cores[g], edgecolor='black', label=f"{g}: média R$ {d[coluna].mean():,.2f} (n={len(d)})", zorder=3)
        plt.axvline(d[coluna].mean(), color=cores[g], linestyle='--', alpha=0.8)
    for _, r in df.iterrows():
        plt.annotate(f"{int(r['id'])}", (r[coluna], 0), textcoords='offset points', xytext=(0, 14), ha='center', fontsize=10)
    plt.yticks([]); plt.xlabel('Preço do ponto de função (R$)'); plt.title(titulo)
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1)
    plt.savefig(arquivo, dpi=150, bbox_inches='tight'); plt.close()
    return df, resumo, sil

seg9, res9, sil9 = segmentar(amostra, 'preco_atualizado_pf', 'Segmentação K-Means: 9 preços Java atualizados pelo IPCA', '6_segmentacao_clusters.png')
print("CENÁRIO PRINCIPAL: 9 preços atualizados"); print(res9); print(f"Índice de silhueta: {sil9:.3f}")
seg9[['id', 'uasg_nome', 'modalidade_nome', 'preco_unitario_pf', 'preco_atualizado_pf', 'Grupo']].sort_values('preco_atualizado_pf')

CENÁRIO PRINCIPAL: 9 preços atualizados
                         count      min      max     mean   median
Grupo                                                             
Grupo 1 (Entrada)            6   558.20   790.01   683.16   685.21
Grupo 2 (Intermediário)      2 1,114.16 1,316.01 1,215.09 1,215.09
Grupo 3 (Premium)            1 2,290.10 2,290.10 2,290.10 2,290.10
Índice de silhueta: 0.639


,id,uasg_nome,modalidade_nome,preco_unitario_pf,preco_atualizado_pf,Grupo
7,8,SUBSECRETARIA DE ASSUNTOS ADMINISTRATIVOS,Pregão - Eletrônico,538.00,558.20,Grupo 1 (Entrada)
1,2,ERN-SECRET.DA SEGURANÇA PUBL. E DEFESA SOCIAL,Inexigibilidade,565.15,597.00,Grupo 1 (Entrada)
0,1,CASA DA MOEDA DO BRASIL/MF,Pregão - Eletrônico,555.00,601.41,Grupo 1 (Entrada)
3,4,UNIVERSIDADE FEDERAL DO CARIRI,Pregão - Eletrônico,735.00,769.02,Grupo 1 (Entrada)
4,5,MINISTÉRIO PÚBLICO DO ESTADO DO PARÁ,Inexigibilidade,750.00,783.30,Grupo 1 (Entrada)
8,9,SECRETARIA MUNICIPAL DA EDUCAÇÃO DE SALVADOR,Inexigibilidade,785.00,790.01,Grupo 1 (Entrada)
2,3,PAGADORIA DE PESSOAL DA MARINHA - COM MARINHA,Pregão - Eletrônico,"1,060.00","1,114.16",Grupo 2 (Intermediário)
6,7,TRIBUNAL DE JUSTIÇA DO ESTADO DO PARÁ,Pregão - Eletrônico,"1,264.22","1,316.01",Grupo 2 (Intermediário)
5,6,INCRA-SEDE/DF,Dispensa,"2,192.73","2,290.10",Grupo 3 (Premium)


In [8]:
# ==========================================
# CÉLULA 7: Cenários de sensibilidade
# ==========================================
cenarios = {
    'C1 - 9 preços (principal)': amostra,
    'C2 - sem outlier IQR (8 preços)': amostra[(amostra['preco_atualizado_pf'] >= li_a) & (amostra['preco_atualizado_pf'] <= ls_a)],
    'C3 - janela de 13 meses (6 preços)': amostra[amostra['dentro_13_meses'] == 'Sim'],
    'C4 - apenas pregões (5 preços)': amostra[amostra['disputa_de_preco'] == 'Sim'],
}
saida = {}
for i, (nome, d) in enumerate(cenarios.items(), 1):
    s = d['preco_atualizado_pf']
    item = {'n': int(len(d)), 'ids': [int(x) for x in d['id']], 'minimo': float(s.min()), 'maximo': float(s.max()),
            'media': float(s.mean()), 'mediana': float(s.median()), 'desvio_padrao': float(s.std()),
            'coef_variacao_pct': float(s.std() / s.mean() * 100)}
    if len(d) >= 5:
        sg, rs, sl = segmentar(d, 'preco_atualizado_pf', f'Segmentação K-Means: {nome}', f'7_cenario_C{i}_clusters.png')
        item['silhueta'] = float(sl)
        item['grupos'] = {g: {'n': int(rs.loc[g, 'count']), 'minimo': float(rs.loc[g, 'min']), 'maximo': float(rs.loc[g, 'max']),
                              'media': float(rs.loc[g, 'mean']), 'mediana': float(rs.loc[g, 'median']),
                              'ids': [int(x) for x in sg[sg['Grupo'] == g]['id']]} for g in NOMES}
    saida[nome] = item
    print(f"\n{nome}: n={item['n']} | média {item['media']:.2f} | mediana {item['mediana']:.2f} | CV {item['coef_variacao_pct']:.1f}%")
    if 'grupos' in item:
        for g, v in item['grupos'].items():
            print(f"   {g}: n={v['n']} | {v['minimo']:.2f} a {v['maximo']:.2f} | média {v['media']:.2f} | itens {v['ids']}")
saida['_nominal'] = {'media': float(series_precos.mean()), 'mediana': float(series_precos.median()), 'desvio_padrao': float(series_precos.std()),
                     'Q1': float(Q1), 'Q3': float(Q3), 'IQR': float(IQR), 'limite_inferior': float(limite_inferior), 'limite_superior': float(limite_superior),
                     'outliers': [float(x) for x in outliers.values], 'media_sem_outliers': float(series_limpa.mean()), 'mediana_sem_outliers': float(series_limpa.median())}
saida['_atualizado'] = {'Q1': float(Q1a), 'Q3': float(Q3a), 'IQR': float(IQRa), 'limite_inferior': float(li_a), 'limite_superior': float(ls_a),
                        'outliers': [float(x) for x in out_at.values]}
json.dump(saida, open('resultados_analise.json', 'w'), ensure_ascii=False, indent=1)


C1 - 9 preços (principal): n=9 | média 979.91 | mediana 783.30 | CV 56.3%
   Grupo 1 (Entrada): n=6 | 558.20 a 790.01 | média 683.16 | itens [1, 2, 4, 5, 8, 9]
   Grupo 2 (Intermediário): n=2 | 1114.16 a 1316.01 | média 1215.09 | itens [3, 7]
   Grupo 3 (Premium): n=1 | 2290.10 a 2290.10 | média 2290.10 | itens [6]



C2 - sem outlier IQR (8 preços): n=8 | média 816.14 | mediana 776.16 | CV 32.9%
   Grupo 1 (Entrada): n=3 | 558.20 a 601.41 | média 585.54 | itens [1, 2, 8]
   Grupo 2 (Intermediário): n=3 | 769.02 a 790.01 | média 780.78 | itens [4, 5, 9]
   Grupo 3 (Premium): n=2 | 1114.16 a 1316.01 | média 1215.09 | itens [3, 7]



C3 - janela de 13 meses (6 preços): n=6 | média 1084.44 | mediana 786.65 | CV 59.2%
   Grupo 1 (Entrada): n=4 | 558.20 a 790.01 | média 725.13 | itens [4, 5, 8, 9]
   Grupo 2 (Intermediário): n=1 | 1316.01 a 1316.01 | média 1316.01 | itens [7]
   Grupo 3 (Premium): n=1 | 2290.10 a 2290.10 | média 2290.10 | itens [6]



C4 - apenas pregões (5 preços): n=5 | média 871.76 | mediana 769.02 | CV 38.0%
   Grupo 1 (Entrada): n=2 | 558.20 a 601.41 | média 579.81 | itens [1, 8]
   Grupo 2 (Intermediário): n=1 | 769.02 a 769.02 | média 769.02 | itens [4]
   Grupo 3 (Premium): n=2 | 1114.16 a 1316.01 | média 1215.09 | itens [3, 7]


In [9]:
# ==========================================
# CÉLULA 8: Comparação com o contrato vigente e com o preço adotado em 2025
# ==========================================
referencias = {'Contrato 014/2022/SEPLAG (05/2025)': 519.99, 'Preço adotado no inventário de 2025': 943.25}
plt.figure(figsize=(12, 6))
ordem = amostra.sort_values('preco_atualizado_pf')
cores = ['#264653' if x == 'Sim' else '#a8a8a8' for x in ordem['disputa_de_preco']]
plt.bar([f"Ref {int(i)}" for i in ordem['id']], ordem['preco_atualizado_pf'], color=cores)
for k, (n, v) in enumerate(referencias.items()):
    plt.axhline(v, linestyle='--', color=['#2a9d8f', '#e76f51'][k], label=f'{n}: R$ {v:,.2f}')
plt.axhline(s_at.median(), linestyle=':', color='black', label=f'Mediana da amostra: R$ {s_at.median():,.2f}')
plt.ylabel('Preço atualizado do ponto de função (R$)'); plt.title('Preços Java atualizados (escuro: pregão; cinza: contratação direta)')
plt.legend(); plt.savefig('8_comparativo_referencias.png', dpi=150, bbox_inches='tight'); plt.close()
print("Gráfico 8 salvo.")

Gráfico 8 salvo.
